In [ ]:
import os, glob, math, random, warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from scipy.ndimage import gaussian_laplace
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import cv2

warnings.filterwarnings("ignore")

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
import os
import glob
import zipfile
from google.colab import drive

# ── MOUNT GOOGLE DRIVE ─────────────────────────────────────────────
drive.mount('/content/drive/')

NUM_CLASS = 6  # change this (1, 2, 3...)
CLASS_NAME = f"class_{NUM_CLASS:02d}"


# ── BASE_DIR ─────────────────────────────────────────────────────

BASE_DIR = f'/content/drive/MyDrive/dataset_ad/{CLASS_NAME}/{CLASS_NAME}'
print("BASE_DIR =", BASE_DIR)

# ── PATHS ───────────────────────────────────────────────────────
TRAIN_GOOD_DIR = os.path.join(BASE_DIR, 'train', 'good')
TRAIN_ANOM_DIRS = glob.glob(os.path.join(BASE_DIR, 'train', 'anomaly_*'))
GT_ANOM_DIRS   = glob.glob(os.path.join(BASE_DIR, 'ground_truth_train', 'anomaly_*'))
TEST_DIR        = os.path.join(BASE_DIR, "test")

# ── Image ─────────────────────────────────────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 16

# ── Depth Anything V2 ─────────────────────────────────────────────────────────
DEPTH_MODEL      = "vits"

# ── Normal map / LoG ──────────────────────────────────────────────────────────
LOG_SIGMA = 1.2   # sigma del Laplacian of Gaussian sulla normal map

# ── PatchCore ─────────────────────────────────────────────────────────────────
CORESET_RATIO = 0.05
K_NEIGHBOURS  = 1

# ── Riproducibilità ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("Config OK")
print(f"  Train good : {TRAIN_GOOD_DIR}")
print(f"  Train anom : {TRAIN_ANOM_DIRS}")
print(f"  GT dirs    : {GT_ANOM_DIRS}")
print(f"  Test dir   : {TEST_DIR}")




In [ ]:

# =============================================================================
# CELL 2 — Dataset & dataloader
# =============================================================================
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

class ImageFolderFlat(Dataset):
    EXTS = {".png", ".jpg", ".jpeg"}

    def __init__(self, items, label, transform=None):
        if isinstance(items, (str, Path)):
            items = [items]
        self.paths = []
        for item in items:
            item = str(item)
            if os.path.isdir(item):
                for p in glob.glob(os.path.join(item, "**", "*"), recursive=True):
                    if os.path.splitext(p)[1].lower() in self.EXTS:
                        self.paths.append(p)
            elif os.path.isfile(item):
                if os.path.splitext(item)[1].lower() in self.EXTS:
                    self.paths.append(item)
        self.paths = sorted(self.paths)
        self.label = label
        self.transform = transform
        print(f"  Found {len(self.paths)} images | label={label}")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, self.label, self.paths[idx]

# ── Split train/val raggruppando per oggetto fisico ───────────────────────────
def get_obj_id(p):
    return os.path.basename(p).split("_view")[0]

all_good = sorted(glob.glob(os.path.join(TRAIN_GOOD_DIR, "*.png")))
groups   = defaultdict(list)
for p in all_good:
    groups[get_obj_id(p)].append(p)

object_ids = list(groups.keys())
train_ids, val_ids = train_test_split(object_ids, test_size=0.2, random_state=SEED)

train_paths    = [p for oid in train_ids for p in groups[oid]]
val_good_paths = [p for oid in val_ids   for p in groups[oid]]

print("Loading datasets...")
train_ds   = ImageFolderFlat(train_paths,    label=0, transform=transform)
val_good   = ImageFolderFlat(val_good_paths, label=0, transform=transform)
test_anom  = ImageFolderFlat(TRAIN_ANOM_DIRS, label=1, transform=transform)
test_ds    = ImageFolderFlat(TEST_DIR,        label=-1, transform=transform)

train_dl      = DataLoader(train_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
val_good_dl   = DataLoader(val_good,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_anom_dl  = DataLoader(test_anom, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_dl       = DataLoader(test_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"\nTrain good  : {len(train_ds)}")
print(f"Val good    : {len(val_good)}")
print(f"Train anom  : {len(test_anom)}")
print(f"Test        : {len(test_ds)}")


In [ ]:
# =============================================================================
# CELL 2b — Download Depth Anything V2 (Google Colab)
# =============================================================================
import os
import sys
import subprocess

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)
    else:
        print(result.stdout)

# ── Clona repo ────────────────────────────────────────────────────────────────
REPO_DIR = "/content/Depth-Anything-V2"

if not os.path.exists(REPO_DIR):
    run(f"git clone https://github.com/DepthAnything/Depth-Anything-V2 {REPO_DIR}")
else:
    print("Repo già presente")

sys.path.insert(0, REPO_DIR)

# ── Dipendenze ────────────────────────────────────────────────────────────────
run("pip install -q huggingface_hub")

# ── Download checkpoint ───────────────────────────────────────────────────────
from huggingface_hub import hf_hub_download

CKPT_DIR = "/content/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

repo_map = {
    "vits": ("depth-anything/Depth-Anything-V2-Small", "depth_anything_v2_vits.pth"),
    "vitb": ("depth-anything/Depth-Anything-V2-Base",  "depth_anything_v2_vitb.pth"),
    "vitl": ("depth-anything/Depth-Anything-V2-Large", "depth_anything_v2_vitl.pth"),
}

repo_id, filename = repo_map[DEPTH_MODEL]
ckpt_path = os.path.join(CKPT_DIR, filename)

if not os.path.isfile(ckpt_path):
    print(f"Scaricando {filename} da HuggingFace...")
    hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        local_dir=CKPT_DIR
    )
    print("Download completato")
else:
    print(f"Checkpoint già presente: {ckpt_path}")

In [ ]:
# =============================================================================
# CELL 3 — Load Depth Anything V2 (Google Colab)
# =============================================================================
import sys
import torch

sys.path.insert(0, "/content/Depth-Anything-V2")

from depth_anything_v2.dpt import DepthAnythingV2

model_configs = {
    "vits": {"encoder": "vits", "features": 64,  "out_channels": [48, 96, 192, 384]},
    "vitb": {"encoder": "vitb", "features": 128, "out_channels": [96, 192, 384, 768]},
    "vitl": {"encoder": "vitl", "features": 256, "out_channels": [256, 512, 1024, 1024]},
}

ckpt_path = f"/content/checkpoints/depth_anything_v2_{DEPTH_MODEL}.pth"

depth_model = DepthAnythingV2(**model_configs[DEPTH_MODEL])

depth_model.load_state_dict(
    torch.load(ckpt_path, map_location="cpu")
)

depth_model = depth_model.to(DEVICE).eval()

print(f"Depth Anything V2 ({DEPTH_MODEL}) loaded")

In [ ]:
# =============================================================================
# CELL 4 — Depth map come input geometrico
# =============================================================================
def image_to_geom_tensor(img_np: np.ndarray) -> torch.Tensor:
    depth = depth_model.infer_image(img_np)
    depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

    if depth.shape[:2] != (IMG_SIZE, IMG_SIZE):
        depth = cv2.resize(depth, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

    depth_3ch = np.stack([depth, depth, depth], axis=-1)
    tensor = torch.from_numpy(depth_3ch).permute(2, 0, 1).float()
    return tensor.unsqueeze(0)

# ── Verifica visiva su 4 immagini anomale ─────────────────────────────────────
sample_paths = random.sample(test_anom.paths, min(4, len(test_anom.paths)))
fig, axes = plt.subplots(2, len(sample_paths), figsize=(4 * len(sample_paths), 6))

for i, path in enumerate(sample_paths):
    img_np = np.array(Image.open(path).convert("RGB"))
    depth  = depth_model.infer_image(img_np)
    depth_vis = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)

    axes[0, i].imshow(img_np)
    axes[0, i].set_title("RGB", fontsize=8)
    axes[0, i].axis("off")

    axes[1, i].imshow(depth_vis, cmap="inferno")
    axes[1, i].set_title("Depth V2", fontsize=8)
    axes[1, i].axis("off")

plt.suptitle("Diagnostica: RGB → Depth V2", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "diagnostic_geom.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Diagnostica salvata in", OUTPUT_DIR)

In [ ]:
# CELL 5 — Feature extractor WideResNet50 (layer 2 + layer 3)
# =============================================================================
torch.backends.cudnn.enabled = False
class WideResNetExtractor(nn.Module):
    """
    WideResNet50 pre-trained, estrae feature da layer2 e layer3.
    Output: lista di tensori (B, C, H, W) — uno per layer.
    """
    def __init__(self):
        super().__init__()
        backbone = models.wide_resnet50_2(weights=models.Wide_ResNet50_2_Weights.IMAGENET1K_V1)
        self.layer0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3

    @torch.no_grad()
    def forward(self, x):
        x = self.layer0(x)
        x = self.layer1(x)
        f2 = self.layer2(x)   # (B, 512, 14, 14) per input 224
        f3 = self.layer3(f2)  # (B,1024,  7,  7)
        return f2, f3

extractor = WideResNetExtractor().to(DEVICE).eval()
print("WideResNet50 extractor pronto")

# ── Dimensioni feature map ─────────────────────────────────────────────────────
with torch.no_grad():
    dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    f2, f3 = extractor(dummy)
print(f"  layer2: {tuple(f2.shape)}   layer3: {tuple(f3.shape)}")


In [ ]:
# =============================================================================
# CELL 6 — Estrazione patch embeddings con salvataggio su disco
# =============================================================================
import os
import gc
import torch
import torch.nn as nn

SAVE_DIR = "/content/patch_embeddings_train"
os.makedirs(SAVE_DIR, exist_ok=True)

def extract_patch_embeddings_to_disk(dataloader, desc="Extracting"):
    """
    Estrae embeddings e li salva batch-per-batch su disco
    evitando overflow RAM.
    """

    saved_files = []
    all_paths = []

    for batch_idx, (batch_imgs, _, batch_paths) in enumerate(dataloader):

        # ── Denormalizzazione ───────────────────────────────────────────────
        imgs_denorm = batch_imgs.clone()

        for c, (m, s) in enumerate(zip(mean, std)):
            imgs_denorm[:, c] = imgs_denorm[:, c] * s + m

        imgs_np = (
            imgs_denorm.permute(0, 2, 3, 1)
            .cpu()
            .numpy() * 255
        ).clip(0, 255).astype(np.uint8)

        batch_features = []

        for img_np, path in zip(imgs_np, batch_paths):

            # ── Geometric feature ──────────────────────────────────────────
            geom_tensor = image_to_geom_tensor(img_np).to(DEVICE)

            with torch.no_grad():

                f2, f3 = extractor(geom_tensor)

                f3_up = nn.functional.interpolate(
                    f3,
                    size=f2.shape[-2:],
                    mode="bilinear",
                    align_corners=False
                )

                combined = torch.cat([f2, f3_up], dim=1)

            B, C, H, W = combined.shape

            patches = (
                combined.permute(0, 2, 3, 1)
                .reshape(-1, C)
                .half()           # usa float16 → metà RAM/disco
                .cpu()
            )

            batch_features.append(patches)
            all_paths.append(path)

            # cleanup GPU
            del geom_tensor, f2, f3, f3_up, combined, patches
            torch.cuda.empty_cache()

        # ── Salvataggio batch ──────────────────────────────────────────────
        batch_tensor = torch.cat(batch_features, dim=0)

        save_path = os.path.join(
            SAVE_DIR,
            f"features_batch_{batch_idx:04d}.pt"
        )

        torch.save(batch_tensor, save_path)

        saved_files.append(save_path)

        # cleanup RAM
        del batch_tensor, batch_features
        gc.collect()

        print(
            f"\r{desc}: {len(all_paths)}/{len(dataloader.dataset)}",
            end=""
        )

    print()

    return saved_files, all_paths


print("Estrazione feature training set...")

train_feature_files, train_paths_used = extract_patch_embeddings_to_disk(
    train_dl,
    desc="Train"
)

print(f"\nSalvati {len(train_feature_files)} file embeddings")

In [ ]:
# =============================================================================
# CELL 7 — Random coreset sampling da disco
# =============================================================================
import random
import gc

def random_coreset_from_disk(feature_files, ratio=CORESET_RATIO):

    sampled_chunks = []
    total_patches = 0
    selected_patches = 0

    # ── Conta patch totali ─────────────────────────────────────────────────
    for f in feature_files:
        x = torch.load(f, map_location="cpu")
        total_patches += x.shape[0]
        del x

    target_total = max(1, int(total_patches * ratio))

    print(f"Coreset target: {total_patches} → {target_total}")

    # ── Sampling batch-by-batch ────────────────────────────────────────────
    remaining_target = target_total
    remaining_total = total_patches

    for f in feature_files:

        x = torch.load(f, map_location="cpu")

        n = x.shape[0]

        # quota proporzionale
        k = max(1, int(n * ratio))

        # evita overshoot finale
        k = min(k, remaining_target)

        idx = torch.randperm(n)[:k]

        sampled = x[idx]

        sampled_chunks.append(sampled)

        selected_patches += k
        remaining_target -= k
        remaining_total -= n

        print(
            f"\rSelected {selected_patches}/{target_total}",
            end=""
        )

        del x, sampled
        gc.collect()

    print()

    memory_bank = torch.cat(sampled_chunks, dim=0)

    return memory_bank


memory_bank = random_coreset_from_disk(
    train_feature_files,
    ratio=CORESET_RATIO
)

memory_bank = memory_bank.to(DEVICE)

print(f"Memory bank shape: {memory_bank.shape}")


In [ ]:
# =============================================================================
# CELL 9 — Anomaly scoring memory-safe
# =============================================================================
import os
import gc
import json
import cv2
import numpy as np
import torch
import torch.nn as nn

SCORES_DIR = "/content/anomaly_scores"
HEATMAP_DIR = "/content/anomaly_heatmaps"

os.makedirs(SCORES_DIR, exist_ok=True)
os.makedirs(HEATMAP_DIR, exist_ok=True)


def anomaly_score_image(img_np: np.ndarray, k: int = K_NEIGHBOURS):

    geom_tensor = image_to_geom_tensor(img_np).to(DEVICE)

    with torch.no_grad():

        f2, f3 = extractor(geom_tensor)

        f3_up = nn.functional.interpolate(
            f3,
            size=f2.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        combined = torch.cat([f2, f3_up], dim=1)

    _, C, H, W = combined.shape

    patches = (
        combined
        .permute(0, 2, 3, 1)
        .reshape(-1, C)
        .to(memory_bank.dtype)
    )

    # ── Chunked cdist per evitare OOM ──────────────────────────────────────
    chunk_size = 512

    patch_scores_all = []

    for i in range(0, patches.shape[0], chunk_size):

        patch_chunk = patches[i:i + chunk_size]

        dists = torch.cdist(
            patch_chunk,
            memory_bank
        )

        nn_dist, _ = dists.topk(
            k,
            largest=False,
            dim=1
        )

        patch_scores = nn_dist.mean(dim=1)

        patch_scores_all.append(
            patch_scores.cpu()
        )

        del patch_chunk, dists, nn_dist, patch_scores
        torch.cuda.empty_cache()

    patch_scores_all = torch.cat(patch_scores_all)

    # ── Heatmap ────────────────────────────────────────────────────────────
    heatmap = (
        patch_scores_all
        .reshape(H, W)
        .float()
        .numpy()
        .astype(np.float32)
    )

    heatmap = cv2.resize(
        heatmap,
        (IMG_SIZE, IMG_SIZE),
        interpolation=cv2.INTER_LINEAR
    )

    score = float(np.percentile(heatmap, 99))

    # cleanup
    del geom_tensor, f2, f3, f3_up, combined, patches
    gc.collect()
    torch.cuda.empty_cache()

    return score, heatmap


def score_dataloader(
    dataloader,
    desc="Scoring",
    save_prefix="dataset"
):

    results = []

    for idx, (batch_imgs, batch_labels, batch_paths) in enumerate(dataloader):

        imgs_denorm = batch_imgs.clone()

        for c, (m, s) in enumerate(zip(mean, std)):
            imgs_denorm[:, c] = imgs_denorm[:, c] * s + m

        imgs_np = (
            imgs_denorm.permute(0, 2, 3, 1)
            .cpu()
            .numpy() * 255
        ).clip(0, 255).astype(np.uint8)

        for img_idx, (img_np, label, path) in enumerate(
            zip(imgs_np, batch_labels.tolist(), batch_paths)
        ):

            score, hmap = anomaly_score_image(img_np)

            # ── Salva heatmap su disco ────────────────────────────────────
            heatmap_path = os.path.join(
                HEATMAP_DIR,
                f"{save_prefix}_{idx:04d}_{img_idx:02d}.npy"
            )

            np.save(heatmap_path, hmap.astype(np.float16))

            results.append({
                "path": path,
                "label": int(label),
                "score": float(score),
                "heatmap": heatmap_path
            })

            print(f"\r{desc}: {len(results)}", end="")

            del hmap
            gc.collect()

    print()

    # ── Salva metadata ─────────────────────────────────────────────────────
    json_path = os.path.join(
        SCORES_DIR,
        f"{save_prefix}_results.json"
    )

    with open(json_path, "w") as f:
        json.dump(results, f)

    return results


print("Scoring val good...")

val_results = score_dataloader(
    val_good_dl,
    desc="Val good",
    save_prefix="val_good"
)

print("Scoring anomalies...")

anom_results = score_dataloader(
    test_anom_dl,
    desc="Train anom",
    save_prefix="anom"
)

In [ ]:
# =============================================================================
# CELL 10 — Verifica visiva heatmap (memory-safe)
# =============================================================================
import math
import gc
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# Usa i risultati salvati nella CELL 9
n_show = len(anom_results)

cols = 8
rows = math.ceil(n_show / cols)

fig, axes = plt.subplots(
    rows * 2,
    cols,
    figsize=(3 * cols, 6 * rows)
)

axes = np.array(axes).reshape(rows * 2, cols)

# Nasconde tutto inizialmente
for ax in axes.flatten():
    ax.axis("off")

for i, result in enumerate(anom_results):

    r = (i // cols) * 2
    c = i % cols

    path = result["path"]
    score = result["score"]

    # ── Load RGB ────────────────────────────────────────────────────────────
    img = np.array(
        Image.open(path)
        .convert("RGB")
        .resize((IMG_SIZE, IMG_SIZE))
    )

    # ── Load heatmap da disco ───────────────────────────────────────────────
    hmap = np.load(result["heatmap"]).astype(np.float32)

    # ── RGB ────────────────────────────────────────────────────────────────
    axes[r, c].imshow(img)

    axes[r, c].set_title(
        f"score={score:.3f}",
        fontsize=7
    )

    axes[r, c].axis("off")

    # ── Overlay heatmap ────────────────────────────────────────────────────
    axes[r + 1, c].imshow(img)

    axes[r + 1, c].imshow(
        hmap,
        cmap="hot",
        alpha=0.7,
        vmin=hmap.min(),
        vmax=hmap.max()
    )

    axes[r + 1, c].set_title(
        "heatmap",
        fontsize=7
    )

    axes[r + 1, c].axis("off")

    # cleanup
    del img, hmap
    gc.collect()

plt.suptitle(
    f"Train anomalie ({n_show} immagini): RGB | Heatmap overlay",
    fontsize=10
)

plt.tight_layout()

# NON salva su disco
plt.show()